In [ ]:
import torch
import numpy as np
import random
import os
from amr.utils import logger
from amr.models import *
import importlib
from torchsummaryX import summary


__all__ = ["init_device", "init_model", "init_loss"]


def init_device(seed=None, cpu=None, gpu=None):
    '''
    配置计算设备和随机种子
    '''
    if seed is not None:

        # 设置随机种子：确保实验结果可复现
        # 对Python内置随机数生成器、PyTorch和NumPy都设置相同的种子
        random.seed(seed)
        torch.manual_seed(seed)
        np.random.seed(seed)
        # 设置cudnn.deterministic为True以确保CUDA操作的结果一致
        torch.backends.cudnn.deterministic = True

    # 如果指定了GPU编号，设置CUDA可见设备
    # 这允许程序只使用指定的GPU
    if gpu is not None:
        os.environ['CUDA_VISIBLE_DEVICES'] = str(gpu)

    # 设备选择逻辑
    # 如果未指定使用CPU且CUDA可用，则使用GPU
    if not cpu and torch.cuda.is_available():
        # 设置设备为CUDA
        device = torch.device('cuda')

        # 启用cuDNN自动调优以寻找最佳卷积算法
        # 注意：这可能会导致结果略有不同，但通常会提高性能
        torch.backends.cudnn.benchmark = True

        # 如果设置了种子，为CUDA操作设置随机种子
        if seed is not None:
            torch.cuda.manual_seed(seed)

        # pin_memory=True：加速CPU到GPU的数据传输
        pin_memory = True

        # 记录日志：当前使用的GPU编号
        logger.info("Running on GPU%d" % (gpu if gpu else 0))

    else:
        # 如果使用CPU或CUDA不可用，则使用CPU
        pin_memory = False
        device = torch.device('cpu')
        logger.info("Running on CPU")

    # 返回设备对象和pin_memory标志
    return device, pin_memory


def init_model(args, network):
    model = getattr(importlib.import_module("amr.models.networks." + args.method + '.' + network), network)(
        len(args.mod_type))
    print(model)
    getattr(importlib.import_module("amr.models.networks." + args.method + '.' + network), "test")()
    if not args.train:
        pretrained = 'results/' + args.method + '/' + network + '/' + args.dataset + '/checkpoints/best_acc.pth'
        assert os.path.isfile(pretrained)
        state_dict = torch.load(pretrained, map_location=torch.device('cpu'))['state_dict']
        model.load_state_dict(state_dict)
        logger.info("pretrained model loaded from {}".format(pretrained))

    return model


def init_loss(loss_func):
    loss = getattr(importlib.import_module("amr.models.losses." + loss_func), loss_func)()
    return loss


if __name__ == '__main__':
    getattr(importlib.import_module("amr.models.networks.ResNet.ResNet"), "test")()
